# 📈 Regresja liniowa - od jednej cechy do wielu

**Plan:**
1. Wczytanie i wstępny wgląd w dane.
2. Regresja z **jedną cechą** (prosta) - najpierw **ręcznie** (sam NumPy), potem **gotowym modelem** (`scikit-learn`).
3. **Ocena jakości** modelu (metryki $R^2$, MSE, RMSE, MAE) i **wykres**.
4. Regresja z **wieloma cechami** (wielowymiarowa/wielokrotna/wieloraka) - również ręcznie i **gotowym modelem**.
5. **Interpretacja parametrów**, podział **train/test** i porównanie modeli.

Model liniowy zakłada, że cel `y` da się przybliżyć **ważoną sumą cech** plus stała:

$$\hat{y} = b_1 x_1 + b_2 x_2 + \dots + b_n x_n + c$$

Dla jednej cechy upraszcza się to do prostej: $\hat{y} = b \cdot x + c$.

Dane: przewidujemy **wynik egzaminu** (`exam_score`) na podstawie **nawyków studentów** (godziny nauki, sen, aktywność itd.).

In [22]:
# Podstawowe biblioteki
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Wczytanie i wstępny wgląd w dane

Zanim zbudujemy model, zawsze warto **zobaczyć dane**: jakie są kolumny, ile jest wierszy, czy nie brakuje wartości.

In [23]:
df = pd.read_csv("2_student_habits_performance.csv")

print("Rozmiar (wiersze, kolumny):", df.shape)
df.head()

Rozmiar (wiersze, kolumny): (1000, 16)


,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,S1000,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,S1001,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,S1002,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,S1003,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,S1004,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


In [24]:
# Typy kolumn i braki danych
# UZUPEŁNIJ - jak sprawdzić podstawowe informację o ramce danych

In [25]:
# Podstawowe statystyki kolumn liczbowych
# UZUPEŁNIJ - jak sprawdzić podstawowe statystyki kolumn liczbowych w ramce danych

### Cel (regresja liniowa prosta)

Zależy nam na przewidzeniu wartości liczbowej zmiennej `y = exam_score`, za pomocą tylko jednej cechy `x = study_hours_per_day`.
Bierzemy je jako **tablice NumPy** (`.to_numpy()`).

In [26]:
x = df["study_hours_per_day"].to_numpy()
y = df["exam_score"].to_numpy()

print("x (5 pierwszych):", x[:5])
print("y (5 pierwszych):", y[:5])
# Jak sprawdzić, czy na pewno długość x i y jest taka sama? 

x (5 pierwszych): [0.  6.9 1.4 1.  5. ]
y (5 pierwszych): [ 56.2 100.   34.3  26.8  66.4]


## 2. Teoria - regresja prosta metodą najmniejszych kwadratów

Szukamy prostej `y = b·x + c`, która leży „najbliżej" wszystkich punktów. „Najbliżej" znaczy: **suma kwadratów błędów** (różnic między prawdziwym a przewidywanym `y`) jest najmniejsza.

Dla jednej cechy istnieje gotowy wzór:

$$b = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2} \qquad\qquad c = \bar{y} - b\,\bar{x}$$

gdzie znak $\Sigma$ oznacza sumę, `x̄`, `ȳ` to średnie.

**Znaczenie parametrów:**
- **`b` (nachylenie / współczynnik)** - o ile średnio zmienia się `y`, gdy `x` wzrośnie o 1. Tu: o ile punktów rośnie wynik za każdą dodatkową godzinę nauki.
- **`c` (wyraz wolny / intercept)** - przewidywana wartość `y`, gdy `x = 0`. Tu: teoretyczny wynik przy zerze godzin nauki.

In [27]:
# Średnie
# UZUPEŁNIJ - jak obliczyć średnią z tablicy numpy
# x_sr = 
# y_sr = 

# Wzór na nachylenie (licznik i mianownik osobno)
# UZUPEŁNIJ - jak obliczyć licznik i mianownik
# licznik = 
# mianownik = 

# Wyznaczenie wzorów na b i c w funkcji liniowej
# b =      # nachylenie
# c =           # wyraz wolny

# print(f"Ręcznie:  y = {b:.4f} * x + {c:.4f}")
# print(f"\nKażda dodatkowa godzina nauki to średnio +{b:.2f} punktu.")
# print(f"Wynik przy 0 h nauki (wyraz wolny): {c:.2f}")

## 3. Predykcja ręczna

Mając `b` i `c`, przewidywanie to zwykłe podstawienie do wzoru `y = b·x + c`. Dzięki wektoryzacji NumPy liczymy od razu dla wielu wartości.

In [28]:
godziny = np.array([2, 5, 8]) # wpisujemy godziny nauki, dla których chcemy przewidzieć wynik egzaminu
# przewidywane = b * godziny + c

# for g, p in zip(godziny, przewidywane):
#     print(f"{g}h nauki  ->  przewidywany wynik: {p:.1f}")

## 4. Jak dobry jest ten model? - metryki oceny

Model zawsze da się utworzyć - pytanie brzmi: **czy w każdym przypadku jest to sensowne?** Do określenia poprawności naszego modelu wprowadzamy pojęcie **METRYKI**. 

> Metryka to miara jakości modelu - liczba mówiąca, jak bardzo przewidywania różnią się od prawdy.

Porównujemy przewidywania `ŷ` z prawdziwymi `y` na wszystkich studentach.

**Najważniejsze metryki:**

- **MAE** (Mean Absolute Error) - średni błąd bezwzględny, w tych samych jednostkach co `y` (punkty). „Średnio mylimy się o tyle punktów."

$$MAE = \frac{1}{n}\sum_{i=1}^{n} \left| y_i - \hat{y}_i \right|$$

- **MSE** (Mean Squared Error) - średni kwadrat błędu. Mocniej karze duże pomyłki.

$$MSE = \frac{1}{n}\sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2$$

- **RMSE** (Root MSE) - pierwiastek z MSE, znów w punktach — łatwiejszy w interpretacji niż MSE.

$$RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2}$$

- **R²** (współczynnik determinacji) - jaką część zmienności `y` model wyjaśnia. `1.0` = idealnie, `0.0` = nie lepiej niż zgadywanie średniej, `<0` = gorzej niż średnia.

$$R^2 = 1 - \frac{\sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2}{\sum_{i=1}^{n} \left( y_i - \bar{y} \right)^2}$$

In [29]:
# Przewidywania modelu na wszystkich danych
# y_hat = b * x + c

# Ręcznie (dla zrozumienia wzorów) - implementacja metryk
# mae = 
# mse = 
# rmse =  
# r2= 

# print("Ręcznie:")
# print(f"  MAE  = {mae:.3f} punktu")
# print(f"  MSE  = {mse:.3f}")
# print(f"  RMSE = {rmse:.3f} punktu")
# print(f"  R2   = {r2:.4f}  ->  model wyjaśnia {r2*100:.1f}% zmienności wyniku")

## 5. Regresja liniowa prosta z wykorzystaniem gotowego modelu (`LinearRegression`)

Teraz dokładnie to samo, ale biblioteką `scikit-learn`. Model sam liczy `b` i `c` - tymi samymi wzorami.
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

> **Uwaga techniczna:** model wymaga, żeby cechy `X` były **dwuwymiarowe** (tabela wiersze × kolumny), dlatego używamy podwójnego nawiasu `df[["study_hours_per_day"]]`.

**Parametry i atrybuty modelu:**
- `LinearRegression(fit_intercept=True)` - `fit_intercept=True` (domyślnie) każe modelowi wyznaczyć wyraz wolny `c`; przy `False` prosta jest zmuszona przejść przez zero.
- `.fit(X, y)` - dopasowuje model (uczy się `b` i `c`).
- `.coef_` - tablica nachyleń (po jednym na cechę); tu `coef_[0]` to nasze `b`.
- `.intercept_` - wyraz wolny `c`.
- `.predict(X)` - zwraca przewidywania.

In [30]:
# scikit-learn - gotowe modele uczenia maszynowego i metryki
# jeśli pojawi się ModuleNotFoundError, zainstaluj:  pip install scikit-learn
from sklearn.linear_model import LinearRegression

X = df[["study_hours_per_day"]] 
y_col = df["exam_score"] 

model = LinearRegression()
model.fit(X, y_col)

b_model = model.coef_[0]
c_model = model.intercept_
print(f"Model:  y = {b_model:.4f} * x + {c_model:.4f}")


Model:  y = 9.4903 * x + 35.9102


In [31]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# To samo funkcjami scikit-learn (dla sprawdzenia)
# print(f"  MAE  = {mean_absolute_error(y, y_hat):.3f}")
# print(f"  RMSE = {np.sqrt(mean_squared_error(y, y_hat)):.3f}")
# print(f"  R2   = {r2_score(y, y_hat):.4f}")

## 6. Wykres - punkty i dopasowana prosta

Najlepszy sposób, żeby zobaczyć, co robi regresja: nanieść dane jako punkty i narysować dopasowaną prostą.

In [32]:
# Prosta do narysowania: od min do max x
xs = np.linspace(x.min(), x.max(), 100)
# ys = b * xs + c

# plt.figure(figsize=(8, 5))
# plt.scatter(x, y, s=10, alpha=0.4, label="studenci") # na danych 
# plt.plot(xs, ys, color="red", linewidth=2, label=f"y = {b:.2f}·x + {c:.2f}") # model
# plt.xlabel("Godziny nauki dziennie")
# plt.ylabel("Wynik egzaminu")
# plt.title(f"Regresja prosta (R2 = {r2:.3f})")
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.show()

## 7. Regresja wielokrotna - wiele cech naraz

Wynik egzaminu może zależeć nie tylko od godzin nauki. Dołóżmy więcej cech: sen, media społecznościowe, obecność itd. Model rozszerza się naturalnie:

$$\hat{y} = b_1 x_1 + b_2 x_2 + \dots + b_n x_n + c$$

Każda cecha `x_i` dostaje swój **własny współczynnik** `b_i`.

**Interpretacja `b_i`:** o ile zmienia się `y`, gdy cecha `x_i` wzrośnie o 1, **przy pozostałych cechach niezmienionych**. To ważna różnica względem zwykłej korelacji - współczynnik "odfiltrowuje" wpływ pozostałych cech.

Zapis macierzowy (`X` z dodaną kolumną jedynek na wyraz wolny):

$$\hat{y} = X\theta \qquad\quad \theta = \begin{bmatrix} c \\ b_1 \\ \vdots \\ b_n \end{bmatrix}$$

### Wybór cech liczbowych

Bierzemy sensowny zestaw cech liczbowych. Kod odfiltrowuje te, których faktycznie nie ma w pliku, i usuwa ewentualne braki danych (`dropna`).

In [33]:
cechy = [
    "study_hours_per_day",
    "sleep_hours",
    "social_media_hours",
    "netflix_hours",
    "attendance_percentage",
    "mental_health_rating",
    "exercise_frequency",
]

# Ramka z cechami + cel, bez braków danych
dane = df[cechy + ["exam_score"]].dropna()

X_multi = dane[cechy].to_numpy()
y_multi = dane["exam_score"].to_numpy()
print("Kształt X:", X_multi.shape, "| Kształt y:", y_multi.shape)

Kształt X: (1000, 7) | Kształt y: (1000,)


### 7a. Ręcznie - równanie normalne

Dla wielu cech istnieje macierzowy odpowiednik równania liniowego: **równanie normalne**:

$$\theta = (X^\top X)^{-1} X^\top y$$

Najpierw dokładamy do `X` kolumnę jedynek - dzięki temu pierwszy element `θ` wyjdzie jako wyraz wolny `c`, a reszta to współczynniki `b_i`.

In [34]:
# Dodaj kolumnę jedynek (na wyraz wolny)
ones = np.ones((X_multi.shape[0], 1))
X_ext = np.hstack([ones, X_multi])

# Równanie normalne:  theta = (XᵀX)^(-1 X^T y
theta = np.linalg.inv(X_ext.T @ X_ext) @ X_ext.T @ y_multi

c_manual = theta[0]
b_manual = theta[1:]

print(f"Wyraz wolny c = {c_manual:.4f}\n")
print("Współczynniki b (ręcznie):")
for nazwa, wsp in zip(cechy, b_manual):
    print(f"  {nazwa:<24} {wsp:+.4f}")

Wyraz wolny c = 6.1572

Współczynniki b (ręcznie):
  study_hours_per_day      +9.5746
  sleep_hours              +2.0046
  social_media_hours       -2.6198
  netflix_hours            -2.2771
  attendance_percentage    +0.1447
  mental_health_rating     +1.9489
  exercise_frequency       +1.4519


### 7b. Ten sam model w scikit-learn

Tu `X` może być zwykłą tablicą 2D - nie trzeba ręcznie dodawać kolumny jedynek, `LinearRegression` sam liczy wyraz wolny.

In [35]:
model_multi = LinearRegression()
model_multi.fit(X_multi, y_multi)

print(f"Wyraz wolny c = {model_multi.intercept_:.4f}\n")
print("Współczynniki b (model):")
for nazwa, wsp in zip(cechy, model_multi.coef_):
    print(f"  {nazwa:<24} {wsp:+.4f}")

print("\nCzy zgodne z wersją ręczną?",
      np.allclose(model_multi.coef_, b_manual) and np.isclose(model_multi.intercept_, c_manual))

Wyraz wolny c = 6.1572

Współczynniki b (model):
  study_hours_per_day      +9.5746
  sleep_hours              +2.0046
  social_media_hours       -2.6198
  netflix_hours            -2.2771
  attendance_percentage    +0.1447
  mental_health_rating     +1.9489
  exercise_frequency       +1.4519

Czy zgodne z wersją ręczną? True


### Jak czytać te współczynniki?

Każdy współczynnik `b_i` to wpływ danej cechy na wynik **przy pozostałych cechach bez zmian**:
- **znak `+`** - cecha podnosi wynik (np. więcej nauki → wyższy wynik),
- **znak `−`** - cecha obniża wynik (np. więcej mediów społecznościowych → niższy wynik),
- **wielkość** - siła wpływu w jednostkach cechy (np. `+2.5` = każda dodatkowa godzina daje +2,5 punktu).

> **Uwaga:** wartości współczynników zależą od **skali cech**. Godziny (0–10) i procent obecności (0–100) są w różnych zakresach, więc samych liczb nie porównuj wprost jako "ważności". Do porównywania ważności stosuje się **standaryzację cech**.

## 8. Ocena i porównanie modeli

Sprawdźmy, czy dołożenie cech faktycznie poprawiło przewidywania względem modelu z jedną cechą.

In [36]:
y_hat_multi = model_multi.predict(X_multi)

r2_multi = r2_score(y_multi, y_hat_multi)
rmse_multi = np.sqrt(mean_squared_error(y_multi, y_hat_multi))
mae_multi = mean_absolute_error(y_multi, y_hat_multi)

# print("Model z JEDNĄ cechą (study_hours_per_day):")
# print(f"  R2   = {r2:.4f}")
# print(f"  RMSE = {rmse:.3f}")
# print(f"  MAE  = {mae:.3f}")

print(f"\nModel z {len(cechy)} cechami:")
print(f"  R2   = {r2_multi:.4f}")
print(f"  RMSE = {rmse_multi:.3f}")
print(f"  MAE  = {mae_multi:.3f}")


Model z 7 cechami:
  R2   = 0.9011
  RMSE = 5.309
  MAE  = 4.204


## 9. Podział na trening i test

Do tej pory liczyliśmy błąd na **tych samych** danych, na których model się uczył. To zbyt optymistyczne - jak sprawdzanie się z odpowiedziami przed sobą. Prawdziwy test to dane, **których model nie widział**.

Dzielimy dane: część do **uczenia** (`train`), część do **testu** (`test`). `random_state` ustala losowanie, żeby wynik był powtarzalny.

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

model_tt = LinearRegression().fit(X_train, y_train)

r2_train = r2_score(y_train, model_tt.predict(X_train))
r2_test  = r2_score(y_test,  model_tt.predict(X_test))

print(f"R2 na treningu: {r2_train:.4f}")
print(f"R2 na teście:   {r2_test:.4f}")

R2 na treningu: 0.9009
R2 na teście:   0.8996


Jeżeli wartości są zbliżone, to oznacza, że model dobrze generalizuje nasze dane - nie przeucza się.

## 10. Predykcja dla nowego studenta

Podajmy hipotetyczne wartości cech i zobaczmy przewidywany wynik. Kolejność wartości musi odpowiadać kolejności listy `cechy`.

In [38]:
# Przykładowy student 
nowy = pd.DataFrame([{
    "study_hours_per_day": 5,
    "sleep_hours": 7,
    "social_media_hours": 2,
    "netflix_hours": 1,
    "attendance_percentage": 90,
    "mental_health_rating": 8,
    "exercise_frequency": 3,
}])[cechy]   # tylko używane cechy, w tej samej kolejności

pred = model_multi.predict(nowy)[0]
print(f"Przewidywany wynik egzaminu: {pred:.1f} punktu")

Przewidywany wynik egzaminu: 93.5 punktu


d:\AI-with-Python\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


## 11. Podsumowanie

- **Regresja liniowa** dopasowuje prostą (lub hiperpłaszczyznę) minimalizującą sumę kwadratów błędów.
- Dla **jednej cechy** liczy się prostym wzorem; dla **wielu cech** - równaniem normalnym / metodą najmniejszych kwadratów. `scikit-learn` robi to za nas jedną linijką, a wyniki są **identyczne** jak ręczne.
- **Parametry:** `b` (nachylenia - wpływ każdej cechy), `c` (wyraz wolny - punkt startowy).
- **Metryki** ($R^2$, RMSE, MAE) mówią, jak dobry jest model; **podział train/test** sprawdza, czy działa na nowych danych.
- Więcej cech zwykle poprawia dopasowanie, ale współczynniki trzeba interpretować ostrożnie (np. różne skale zmiennych).

# ZADANIA

## Zadanie 2 - Regresja prosta: czy oceny krytyków przewidują sprzedaż?

Dane: `2_cw_Video_Games_Sales.csv` (sprzedaż gier komputerowych).
Cel (`y`): `Global_Sales` - sprzedaż globalna w milionach egzemplarzy.
Cecha (`x`): `Critic_Score` - ocena krytyków.

**Do zrobienia:**
1. Wczytaj dane i zostaw tylko wiersze, w których `Critic_Score` oraz
   `Global_Sales` mają wartości (użyj `dropna` na tych dwóch kolumnach).
2. Policz współczynniki prostej `y = b·x + c` **ręcznie**, korzystając ze wzoru
   najmniejszych kwadratów (średnie, licznik, mianownik).
3. Sprawdź wynik gotowym modelem `LinearRegression` - `b` i `c` powinny się zgadzać.
4. Narysuj wykres: punkty (gry) + dopasowana prosta.
5. Policz **$R^2$** dla tego modelu.

## Zadanie 3 - Regresja wielowymiarowa: czy więcej cech pomoże?

Dane i cel jak wyżej (`y = Global_Sales`).
Cechy (`X`): `Critic_Score`, `User_Score`, `Critic_Count`, `User_Count`.

**Uwaga:** kolumna `User_Score` jest zapisana jako tekst i zawiera
wartość `"tbd"`. Zamień ją na liczby: `pd.to_numeric(df["User_Score"], errors="coerce")`,
a następnie usuń braki (`dropna`) na wszystkich używanych kolumnach.

**Do zrobienia:**
1. Przygotuj `X` (cztery cechy) oraz `y`, po oczyszczeniu danych.
2. Dopasuj model `LinearRegression` na wielu cechach.
3. Wypisz współczynnik `b` dla każdej cechy oraz wyraz wolny `c`.
4. Policz **$R^2$** i **RMSE**, i porównaj je z modelem z Zadania 2 (jedna cecha).
5. Podziel dane na train/test (`train_test_split`, `test_size=0.2`,
   `random_state=42`) i sprawdź $R^2$ osobno na treningu i teście.